# `train_lstm.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [ ]:
import os
import random
import sys

# notebook 没有 __file__：同时把「当前目录」和「上一级目录」加入 sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for _p in (os.getcwd(), PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import torch

from neural_config import *
from neural_data import (
    load_model_data,
    make_expanding_splits,
    make_fold_datasets,
)
from neural_models import LSTMDirect
from neural_backtest import (
    predict_dataset,
    build_daily_portfolio,
    rescale_to_target_vol,
    summary_table,
)
from neural_search import fit_ensemble

### Reproducibility & device

In [2]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


### Load data & build expanding-window splits

In [3]:
df = load_model_data(DATA_PATH)

splits = make_expanding_splits(df)

all_predictions = []
search_logs = []

Loaded 240,991 rows | 51 tickers | 2002-04-08 -> 2026-09-02


### Walk-forward training loop (one model per fold)

In [4]:
for fold, split in enumerate(splits, 1):

    print("\n" + "=" * 70)

    print(
        f"FOLD {fold}: "
        f"train <= {split['train_end']} | "
        f"test {split['test_start']}-{split['test_end']}"
    )

    train_ds, val_ds, test_ds = make_fold_datasets(
        df, split, lookback=LSTM_LOOKBACK, validation_fraction=VAL_FRACTION
    )

    print(
        f"train={len(train_ds):,} "
        f"val={len(val_ds):,} "
        f"test={len(test_ds):,}"
    )

    def build_model(cfg):
        return LSTMDirect(n_features=len(FEATURES), hidden_size=cfg["hidden_size"], num_layers=1, dropout=cfg["dropout"])

    models, search_log = fit_ensemble(
        build_model,
        train_ds,
        val_ds,
        device=device,
        fold=fold,
        tag="LSTM",
        search_iter=SEARCH_ITER_LSTM,
    )

    search_logs.append(search_log)

    predictions = predict_dataset(
        models,
        test_ds,
        device,
    )

    predictions["fold"] = fold

    all_predictions.append(predictions)


FOLD 1: train <= 2006 | test 2007-2011
train=15,780 val=2,040 test=36,892
[LSTM fold 1] search 1/10 val_sharpe=-2.383 train_sharpe=-0.540 epoch=1 | {'hidden_size': 10, 'dropout': 0.2, 'batch_size': 1024, 'learning_rate': 0.0001, 'max_grad_norm': 1.0, 'weight_decay': 0.0001}
[LSTM fold 1] search 2/10 val_sharpe=+2.134 train_sharpe=+0.692 epoch=16 | {'hidden_size': 40, 'dropout': 0.4, 'batch_size': 1024, 'learning_rate': 0.0001, 'max_grad_norm': 1.0, 'weight_decay': 0.001}
[LSTM fold 1] search 3/10 val_sharpe=-1.627 train_sharpe=-0.251 epoch=30 | {'hidden_size': 10, 'dropout': 0.3, 'batch_size': 2048, 'learning_rate': 0.0001, 'max_grad_norm': 10.0, 'weight_decay': 0.001}
[LSTM fold 1] search 4/10 val_sharpe=+2.099 train_sharpe=+0.618 epoch=2 | {'hidden_size': 20, 'dropout': 0.1, 'batch_size': 512, 'learning_rate': 0.001, 'max_grad_norm': 1.0, 'weight_decay': 0.001}
[LSTM fold 1] search 5/10 val_sharpe=+2.227 train_sharpe=+0.766 epoch=17 | {'hidden_size': 80, 'dropout': 0.4, 'batch_size'

### Aggregate OOS predictions

In [5]:
pred = pd.concat(
    all_predictions,
    ignore_index=True,
)

pd.concat(search_logs, ignore_index=True).to_csv(
    "lstm_search_log.csv",
    index=False,
)

pred.to_csv(
    "lstm_oos_predictions.csv",
    index=False,
)

### Daily portfolio returns & performance report

In [6]:
daily = build_daily_portfolio(pred)

raw_ret = daily["port_ret"].dropna()

rescaled_ret = rescale_to_target_vol(
    raw_ret
).dropna()


print("\n=== LSTM DIRECT SHARPE — RAW ===")

for k, v in summary_table(raw_ret).items():
    print(f"{k:<20s} {v:.4f}")


print("\n=== LSTM DIRECT SHARPE — 15% PORTFOLIO VOL ===")

for k, v in summary_table(rescaled_ret).items():
    print(f"{k:<20s} {v:.4f}")


daily["raw_return"] = raw_ret

daily["rescaled_return"] = rescaled_ret

daily.to_csv(
    "lstm_oos_daily_returns.csv"
)

print("\nSaved:")
print("  lstm_search_log.csv")
print("  lstm_oos_predictions.csv")
print("  lstm_oos_daily_returns.csv")


=== LSTM DIRECT SHARPE — RAW ===
E[Return]            0.0171
Vol.                 0.0262
Downside Deviation   0.0229
MDD                  0.0734
Sharpe               0.6510
Sortino              0.7454
Calmar               0.2328
% +ve Returns        0.5485
Ave.P/Ave.L          0.9329

=== LSTM DIRECT SHARPE — 15% PORTFOLIO VOL ===
E[Return]            0.1005
Vol.                 0.1564
Downside Deviation   0.1250
MDD                  0.2901
Sharpe               0.6427
Sortino              0.8037
Calmar               0.3464
% +ve Returns        0.5485
Ave.P/Ave.L          0.9237

Saved:
  lstm_search_log.csv
  lstm_oos_predictions.csv
  lstm_oos_daily_returns.csv
